# CCGT NPV simulation

Run the CCGT electricity Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity.electricity_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_SAMPLE_SIZE,
    simulate_electricity_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'ccgt'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED

results_by_technology = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,annual_output_mwh,full_load_hours_per_year,lifetime_years,capacity_mw,capacity_kw,capex_eur_per_kw,fixed_opex_eur_per_kw_year,variable_opex_eur_per_mwh,...,annual_variable_opex_eur,annual_fuel_cost_eur,annual_emissions_cost_eur,annual_total_cost_eur,annual_net_cash_flow_eur,npv_eur,discounted_lifetime_output_mwh,present_value_total_cost_eur,lcoe_eur_per_mwh,levelized_net_margin_eur_per_mwh
0,0,ccgt,1000000.0,4650.0,30.0,215.053763,215053.763441,1288.983500,20.590748,4.991211,...,4.991211e+06,1.001273e+08,2.716885e+07,1.367155e+08,-4.264547e+07,-7.572942e+08,1.125778e+07,1.816314e+09,161.338500,-67.268500
1,1,ccgt,1000000.0,4650.0,30.0,215.053763,215053.763441,1298.581244,19.475981,4.551998,...,4.551998e+06,5.033909e+07,2.716480e+07,8.624427e+07,7.825735e+06,-1.911644e+08,1.125778e+07,1.250184e+09,111.050639,-16.980639
2,2,ccgt,1000000.0,4650.0,30.0,215.053763,215053.763441,957.080860,20.312246,4.925043,...,4.925043e+06,5.280665e+07,2.679264e+07,8.889255e+07,5.177447e+06,-1.475373e+08,1.125778e+07,1.206557e+09,107.175357,-13.105357
3,3,ccgt,1000000.0,4650.0,30.0,215.053763,215053.763441,902.628894,19.405181,5.836702,...,5.836702e+06,5.693228e+07,2.700822e+07,9.395036e+07,1.196352e+05,-1.927669e+08,1.125778e+07,1.251787e+09,111.192990,-17.122990
4,4,ccgt,1000000.0,4650.0,30.0,215.053763,215053.763441,1024.561025,23.241287,5.533202,...,5.533202e+06,3.436273e+07,2.709113e+07,7.198519e+07,2.208481e+07,2.829032e+07,1.125778e+07,1.030729e+09,91.557043,2.512957


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoe_eur_per_mwh = results["lcoe_eur_per_mwh"]
levelized_net_margin_eur_per_mwh = results["levelized_net_margin_eur_per_mwh"]
summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoe_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOE EUR/MWh"
        ),
        levelized_net_margin_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/MWh"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOE EUR/MWh,Levelized net margin EUR/MWh
count,100000.000000,100000.000000,100000.000000
mean,-325.132901,122.950721,-28.880721
std,266.260931,23.651275,23.651275
min,-1257.323103,74.532489,-111.684784
5%,-808.454499,88.551999,-71.812938
50%,-297.741161,120.517583,-26.447583
95%,62.120462,165.882938,5.518001
max,219.949062,205.754784,19.537511


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),10596,0.10596
1,Negative (NPV < 0),89404,0.89404


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"CCGT NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()

/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42900/1466186080.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## levelized net margin histogram


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_mwh,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"CCGT levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/MWh)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42900/723580808.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")

annual_revenue_eur           94.070000
annual_fixed_opex_eur         4.444431
annual_variable_opex_eur      5.164574
annual_fuel_cost_eur         65.363966
annual_emissions_cost_eur    26.960102
annual_net_cash_flow_eur     -7.863073
Name: Mean annual value, million EUR, dtype: float64